In [1]:
setwd("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/frontal_cortex")

library(dplyr)
library(WGCNA)
library(data.table)

source("/mnt/lareaulab/reliscu/code/FindModules/FindModules.R")


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: dynamicTreeCut

Loading required package: fastcluster


Attaching package: ‘fastcluster’


The following object is masked from ‘package:stats’:

    hclust





Attaching package: ‘WGCNA’


The following object is masked from ‘package:stats’:

    cor



Attaching package: ‘data.table’


The following objects are masked from ‘package:dplyr’:

    between, first, last



Attaching package: ‘flashClust’


The following object is masked from ‘package:fastcluster’:

    hclust


The following object is masked from ‘package:stats’:

    hclust



Attaching package: ‘svMisc’


The following object is masked from ‘package:utils’:

    ?


Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:dplyr’:



Allowing multi-threading with up to 48 threads.


Here I run FM to (hopefully) find modules representing each of the cell types present in the single-cell data

In [ ]:
data_source <- "GTEx_frontal_cortex_counts_TMMF_All_200_outliers_removed"
expr <- fread("GTEx_frontal_cortex_counts_TMMF_SampleNetworks/All_10-58-02/GTEx_frontal_cortex_counts_TMMF_All_200_outliers_removed.csv", data.table=FALSE)
colnames(expr)[1] <- "Gene"

sampleinfo <- read.csv("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/bulk/GTEx/cortex/GTEx_cortex_sampleinfo.csv")

ERROR: Error in fread("GTEx_frontal_cortex_counts_TMMF_SampleNetworks/All_10-58-02/GTEx_frontal_cortex_counts_TMMF_All_200_outliers_removed.csv", : could not find function "fread"


In [ ]:
# Subset to genes in the top X percentile

not_ribo_genes <- !grepl("^RP", expr[,1])
not_mito_genes <- !grepl("^MT−", expr[,1])

prob <- .6
mean_expr <- rowMeans(expr[,-1])

min_cutoff <- unname(quantile(mean_expr, prob))
max_cutof <- unname(quantile(mean_expr, .99))

print(paste("quantile(mean_expr, prob):", round(min_cutoff, 3)))

subset <- (mean_expr >= min_cutoff) & (mean_expr < max_cutof) & not_ribo_genes & not_mito_genes
sum(subset)

[1] "quantile(mean_expr, prob): 68.768"


[1] 15915

In [ ]:
# Order samples by covariates of interest

sampleinfo[,1] <- make.names(sampleinfo[,1])
sampleinfo <- sampleinfo[sampleinfo[,1] %in% colnames(expr),]
sampleinfo$Mean_age <- sapply(strsplit(sampleinfo$AGE, "-"), function(x) mean(as.numeric(x)))
sampleinfo$SAMPID <- make.names(sampleinfo$SAMPID)

sampleinfo <- sampleinfo %>% arrange(Mean_age)
    
expr <- expr[, c(1, match(sampleinfo[,1], colnames(expr)[-1]) + 1)]
all.equal(sampleinfo[,1], colnames(expr)[-1])

In [ ]:
samplegroups <- as.factor(sampleinfo$AGE)
merge.param <- 0.95
projectname <- paste0(data_source, "_mergeParam", merge.param, "_subsetCutoff", round(min_cutoff, 3))

In [ ]:
FindModules(
  projectname=projectname,
  expr=expr,
  geneinfo=1,
  sampleindex=2:ncol(expr),
  samplegroups=samplegroups,
  subset=subset,
  simMat=NULL,
  saveSimMat=FALSE,
  simType="Bicor",
  beta=1,
  overlapType="None",
  TOtype="signed",
  TOdenom="min",
  MIestimator="mi.mm",
  MIdisc="equalfreq",
  signumType="rel",
  iterate=TRUE,
  signumvec=rev(c(.95,.94,.93,.92,.91,.9)), 
  minsizevec=c(4,5,6,8,10), 
  signum=NULL,
  minSize=NULL,
  merge.by="ME",
  merge.param=merge.param,
  export.merge.comp=T,
  ZNCcut=2,
  calcSW=FALSE,
  loadTree=FALSE,
  writeKME=TRUE,
  calcBigModStat=FALSE,
  writeModSnap=TRUE
)